# Clustering Evaluation Metrics - Visualization

Plot silhouette score, Davies-Bouldin index, within-cluster sum of squares (inertia), and ARI stability across k values (k = 2–15).

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

In [8]:
load_dotenv()
base_dir = os.environ['BASE_DIR']
results_dir = os.environ['RESULTS_DIR']
runs_str = "run-ON"
metrics_dir = os.path.join(base_dir, f"coactivation_patterns_{runs_str}", f"eval_metrics")
ari_dir = os.path.join(metrics_dir, "ari_stability")
results_fig_dir = os.path.join(results_dir, "supplementary_figures")

In [9]:
# Define k range
k_range = range(2, 16)
k_values = list(k_range)

# Load existing metrics
silhouette_mean = np.load(os.path.join(metrics_dir, 'silhouette_mean.npy'))
silhouette_std = np.load(os.path.join(metrics_dir, 'silhouette_std.npy'))
davies_mean = np.load(os.path.join(metrics_dir, 'davies_mean.npy'))
davies_std = np.load(os.path.join(metrics_dir, 'davies_std.npy'))
inertia_mean = np.load(os.path.join(metrics_dir, 'inertia_mean.npy'))
inertia_std = np.load(os.path.join(metrics_dir, 'inertia_std.npy'))

In [10]:
# Load ARI stability scores
ari_data = np.load(os.path.join(ari_dir, 'ari_stability_scores.npy'))
ari_mean = {int(k): float(score) for k, score in ari_data}

# Calculate std from individual scores
ari_std = {}
for k in k_values:
    all_scores = np.load(os.path.join(ari_dir, f'ari_all_scores_k{k}.npy'))
    ari_std[k] = np.std(all_scores)

# Convert ARI dict to arrays
ari_mean_array = np.array([ari_mean[k] for k in k_values])
ari_std_array = np.array([ari_std[k] for k in k_values])

In [ ]:
from matplotlib.gridspec import GridSpec

# Load CAP_TS and compute R-squared increases for the 5th subplot
output_dir = os.path.join(base_dir, f"coactivation_patterns_{runs_str}")
CAP_TS = np.load(os.path.join(output_dir, f"CAP_TS_{runs_str}_zscored.npy"))
threshold = 0.01
tss = np.sum((CAP_TS - np.mean(CAP_TS, axis=0)) ** 2)
r_squared_values = 1 - (inertia_mean / tss)
r_squared_increases = np.diff(r_squared_values)

# Define consistent color and line properties
plot_color = '#2E86AB'
line_width = 3
marker_size = 8
k_annotate = [5, 6, 7]

# 3-row x 4-col grid: top two rows = 4 plots (2 cols each), bottom row = 1 centered plot
fig = plt.figure(figsize=(12, 15))
gs = GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.55, left=0.08, right=0.92)

ax1 = fig.add_subplot(gs[0, 0:2])   # Silhouette score
ax2 = fig.add_subplot(gs[0, 2:4])   # Davies-Bouldin index
ax3 = fig.add_subplot(gs[1, 0:2])   # Within-cluster sum of squares
ax4 = fig.add_subplot(gs[1, 2:4])   # Adjusted Rand index
ax5 = fig.add_subplot(gs[2, 1:3])   # R-squared increases (centered)

# Plot 1: Silhouette score
ax1.errorbar(k_values, silhouette_mean, yerr=silhouette_std,
             fmt='o-', capsize=5, capthick=2, linewidth=line_width, markersize=marker_size,
             color=plot_color, ecolor=plot_color, alpha=0.8)
text_str = '\n'.join([f'k = {k}, mean = {silhouette_mean[k-2]:.3f}' for k in k_annotate])
ax1.text(0.95, 0.95, text_str, transform=ax1.transAxes,
         fontsize=12, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax1.set_xlabel('Number of clusters (k)', fontsize=12)
ax1.set_ylabel('Silhouette score', fontsize=12)
ax1.set_title('Silhouette score', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.tick_params(labelsize=11)

# Plot 2: Davies-Bouldin index
ax2.errorbar(k_values, davies_mean, yerr=davies_std,
             fmt='o-', capsize=5, capthick=2, linewidth=line_width, markersize=marker_size,
             color=plot_color, ecolor=plot_color, alpha=0.8)
text_str = '\n'.join([f'k = {k}, mean = {davies_mean[k-2]:.3f}' for k in k_annotate])
ax2.text(0.95, 0.05, text_str, transform=ax2.transAxes,
         fontsize=12, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax2.set_xlabel('Number of clusters (k)', fontsize=12)
ax2.set_ylabel('Davies-Bouldin index', fontsize=12)
ax2.set_title('Davies-Bouldin index', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.tick_params(labelsize=11)

# Plot 3: Within-cluster sum of squares
ax3.errorbar(k_values, inertia_mean, yerr=inertia_std,
             fmt='o-', capsize=5, capthick=2, linewidth=line_width, markersize=marker_size,
             color=plot_color, ecolor=plot_color, alpha=0.8)
text_str = '\n'.join([f'k = {k}, mean = {inertia_mean[k-2]:.1f}' for k in k_annotate])
ax3.text(0.95, 0.95, text_str, transform=ax3.transAxes,
         fontsize=12, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax3.set_xlabel('Number of clusters (k)', fontsize=12)
ax3.set_ylabel('Inertia (WCSS)', fontsize=12)
ax3.set_title('Within-cluster sum of squares', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.tick_params(labelsize=11)

# Plot 4: Adjusted Rand index
ax4.errorbar(k_values, ari_mean_array, yerr=ari_std_array,
             fmt='o-', capsize=5, capthick=2, linewidth=line_width, markersize=marker_size,
             color=plot_color, ecolor=plot_color, alpha=0.8)
text_str = '\n'.join([f'k = {k}, mean = {ari_mean_array[k-2]:.3f}' for k in k_annotate])
ax4.text(0.95, 0.95, text_str, transform=ax4.transAxes,
         fontsize=12, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax4.set_xlabel('Number of clusters (k)', fontsize=12)
ax4.set_ylabel('ARI score', fontsize=12)
ax4.set_title('Adjusted Rand index', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.tick_params(labelsize=11)

# Plot 5: R-squared increases (centered)
bar_colors = [plot_color if inc > threshold else 'lightgray' for inc in r_squared_increases]
ax5.bar(k_values[1:], r_squared_increases, color=bar_colors, alpha=0.8)
ax5.axhline(y=threshold, color='r', linestyle='--', linewidth=2,
            label=f'Threshold = {threshold:.2f}')
ax5.set_xlabel('Number of clusters (k)', fontsize=12)
ax5.set_ylabel('R-squared increase', fontsize=12)
ax5.set_title("R-squared increases between consecutive k's", fontsize=14, fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.tick_params(labelsize=11)
ax5.legend(fontsize=11)

# Save figure
output_path = os.path.join(results_fig_dir, '01_clustering_eval_metrics.png')
plt.savefig(output_path, dpi=400, bbox_inches='tight')

plt.show()
plt.close()

## R-squared Analysis

R-squared (explained variance) per k and the marginal increase between consecutive k values.

In [ ]:
# Load CAP_TS to compute total sum of squares
output_dir = os.path.join(base_dir, f"coactivation_patterns_{runs_str}")
CAP_TS = np.load(os.path.join(output_dir, f"CAP_TS_{runs_str}_zscored.npy"))

# Compute R-squared and increases
threshold = 0.01
total_mean = np.mean(CAP_TS, axis=0)
tss = np.sum((CAP_TS - total_mean) ** 2)

r_squared_values = 1 - (inertia_mean / tss)
r_squared_std = inertia_std / tss
r_squared_increases = np.diff(r_squared_values)

# Find optimal k (last k with significant increase + 1)
significant_increases = r_squared_increases > threshold
try:
    optimal_k = np.where(significant_increases)[0][-1] + 3
except IndexError:
    optimal_k = 2

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), height_ratios=[2, 1])

# Plot R-squared with error bars
ax1.errorbar(k_values, r_squared_values, yerr=r_squared_std,
             fmt='o-', capsize=5, capthick=2, linewidth=line_width, markersize=marker_size,
             color=plot_color, ecolor=plot_color, alpha=0.8)
ax1.axvline(x=optimal_k, color='r', linestyle='--', linewidth=2,
            label=f'Optimal k = {optimal_k}')
text_str = '\n'.join([f'k = {k}, R² = {r_squared_values[k-2]:.3f}' for k in k_annotate])
ax1.text(0.95, 0.05, text_str, transform=ax1.transAxes,
         fontsize=12, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax1.set_xlabel('Number of clusters (k)', fontsize=12)
ax1.set_ylabel('R-squared', fontsize=12)
ax1.set_title('R-squared vs number of clusters', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.tick_params(labelsize=11)
ax1.legend(fontsize=11)

# Plot R-squared increases as bar chart
bar_colors = [plot_color if inc > threshold else 'lightgray' for inc in r_squared_increases]
ax2.bar(k_values[1:], r_squared_increases, color=bar_colors, alpha=0.8)
ax2.axhline(y=threshold, color='r', linestyle='--', linewidth=2,
            label=f'Threshold = {threshold:.2f}')
ax2.set_xlabel('Number of clusters (k)', fontsize=12)
ax2.set_ylabel('R-squared increase', fontsize=12)
ax2.set_title("R-squared increases between consecutive k's", fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.tick_params(labelsize=11)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()
plt.close()